# Clean first stage: tripartite cycle separators

## Purpose

This notebook carries out the first, deliberately self-contained stage of the project. The object of interest is the **full tripartite cycle polytope**

$$
\mathcal{C}_{\mathrm{cycle}}
=
\operatorname{conv}\!\left(
\mathcal{V}_{\circlearrowright}
\cup
\mathcal{V}_{\circlearrowleft}
\right),
$$

where the two vertex sets correspond to the clockwise and counterclockwise communication patterns. The essential correction relative to the earlier implementation is that these two orientations must be generated by genuinely different dependency rules; the counterclockwise list must not accidentally duplicate the clockwise one.

For each of the four nonsignalling extremal representatives in classes 27, 29, 43, and 44, the notebook constructs a linear Bell functional $c$ such that

$$
c\cdot p \geq 1
\qquad\text{for every }p\in\mathcal{C}_{\mathrm{cycle}},
$$

while the chosen nonsignalling point $q$ has $c\cdot q<1$. Such a functional separates $q$ from the classical-with-communication polytope.

## What is done, in order

1. Fix one explicit coordinate convention for the 64 probabilities $p(abc\mid xyz)$.
2. Generate and audit the deterministic vertices of both cycle orientations.
3. Reconstruct the four published nonsignalling representatives from their correlators.
4. Verify exactly that each target is a normalized, nonsignalling extremal point.
5. Solve one separation LP per target against the corrected union of all deterministic vertices.
6. Rationalize and independently check each inequality, then print its coefficients as an $8\times8$ matrix.

This is intentionally only the first stage. It contains **no** perturbation/mixing search, NPA calculation, or see-saw optimization. Those later steps should consume the audited inequalities produced here.

## Requirements and execution

The code is written in Julia and uses `LinearAlgebra`, `Printf`, `JuMP`, and `GLPK`. Execute the notebook from top to bottom: later cells intentionally depend on objects and checks created by earlier cells. Assertions are part of the audit; if one fails, execution should stop rather than silently continue with inconsistent coordinates or geometry.


## 1. Canonical probability coordinates

A behavior in the binary-input, binary-output tripartite scenario consists of

$$
p(abc\mid xyz),
\qquad a,b,c,x,y,z\in\{0,1\}.
$$

It is stored as a vector of length $2^6=64$. To remove any ambiguity, the vector is split into eight input blocks, ordered lexicographically as

$$
xyz=000,001,010,011,100,101,110,111.
$$

Inside each input block, the eight outputs use the same order,

$$
abc=000,001,010,011,100,101,110,111.
$$

Thus $z$ is the fastest-changing input bit and $c$ is the fastest-changing output bit. In Julia's one-based indexing, the coordinate of $p(abc\mid xyz)$ is

$$
1+c+2b+4a+8z+16y+32x.
$$

The binary outputs are also interpreted as signs when correlators are used later: output `0` corresponds to $+1$, and output `1` corresponds to $-1$.

The two assertions at the end of the next cell are small but important regression tests. They verify both the within-block output order and the order of the eight input blocks. Keeping this map explicit prevents the kind of silent iterator-order error that can change a polytope while leaving array dimensions apparently correct.


In [3]:
using LinearAlgebra
using Printf
using JuMP
using GLPK

#const MOI = JuMP.MathOptInterface
const CLASSES = (27, 29, 43, 44)
const TRIPLES = [(u, v, w) for u in 0:1 for v in 0:1 for w in 0:1]

#
# The 64-vector is ordered in blocks of fixed (x,y,z):
#
#   (x,y,z) = 000, 001, 010, 011, 100, 101, 110, 111.
#
# Within each block, the output order is
#
#   (a,b,c) = 000, 001, 010, 011, 100, 101, 110, 111.
#
# Thus c and z are the fastest-changing indices. Binary output 0 corresponds
# to the physicists' value +1, and binary output 1 corresponds to -1.

prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

@assert [prob_index(a, b, c, 0, 0, 0) for (a, b, c) in TRIPLES] == collect(1:8)
@assert [prob_index(0, 0, 0, x, y, z) for (x, y, z) in TRIPLES] == collect(1:8:57)




## 2. Correct deterministic vertices of the two cycle orientations

Each party uses a deterministic binary response function of two binary inputs. There are

$$
2^{2^2}=16
$$

such functions. The integer masks `0:15` encode their four-entry truth tables; `response(mask,u,v)` reads the output assigned to the pair $(u,v)$.

The two communication orientations have different causal dependencies:

| orientation | Alice | Bob | Charlie |
|---|---|---|---|
| clockwise | $a=f_A(x,z)$ | $b=f_B(y,x)$ | $c=f_C(z,y)$ |
| counterclockwise | $a=f_A(x,y)$ | $b=f_B(y,z)$ | $c=f_C(z,x)$ |

Choosing one of 16 response functions independently for each party gives $16^3=4096$ deterministic behaviors per orientation. Every deterministic behavior has exactly one probability equal to 1 in each input block and all other entries equal to 0.

The polytope used by the LP is the convex hull of the **union** of the two orientation lists. Their intersection contains the 64 fully local deterministic behaviors, so removing duplicate rows gives

$$
4096+4096-64=8128
$$

distinct vertices. Removing duplicates does not change the convex hull or any LP optimum, but it exposes the correct combinatorics and avoids redundant constraints.

This cell performs structural checks before any inequality is computed: binary entries, one selected output per input, expected vertex counts, intersection size, and affine dimensions. The individual orientation polytopes have affine dimension 44; their corrected union has affine dimension 50. In particular, the jump from 44 to 50 is a strong diagnostic that the second orientation was not accidentally generated as a copy of the first.


In [5]:

"""
    response(mask, u, v)

Evaluate one of the 16 deterministic binary functions of two binary inputs.
Bit `2u + v` of `mask` stores the output for the input pair `(u,v)`.
"""
function response(mask::Int, u::Int, v::Int)
    @assert 0 <= mask <= 15
    @assert u in 0:1 && v in 0:1
    return Int((mask >> (2*u + v)) & 1)
end

"""
    deterministic_vertex(fA, fB, fC, orientation)

Construct one deterministic 64-component behavior in canonical coordinates.

For `:clockwise`:
    a = fA(x,z),  b = fB(y,x),  c = fC(z,y).

For `:counterclockwise`:
    a = fA(x,y),  b = fB(y,z),  c = fC(z,x).
"""
function deterministic_vertex(
    fA::Int,
    fB::Int,
    fC::Int,
    orientation::Symbol,
)
    vertex = zeros(Int8, 64)

    for (x, y, z) in TRIPLES
        if orientation === :clockwise
            a = response(fA, x, z)
            b = response(fB, y, x)
            c = response(fC, z, y)
        elseif orientation === :counterclockwise
            a = response(fA, x, y)
            b = response(fB, y, z)
            c = response(fC, z, x)
        else
            error("Unknown orientation: $orientation")
        end

        vertex[prob_index(a, b, c, x, y, z)] = 1
    end

    return vertex
end

"""Generate the 4096 deterministic vertices of one cycle orientation."""
function generate_orientation(orientation::Symbol)
    vertices = Matrix{Int8}(undef, 16^3, 64)
    row = 1

    for fA in 0:15, fB in 0:15, fC in 0:15
        vertices[row, :] .= deterministic_vertex(fA, fB, fC, orientation)
        row += 1
    end

    return vertices
end

row_set(vertices) = Set(Tuple(row) for row in eachrow(vertices))

function affine_dimension(points; atol=1e-9)
    n = size(points, 1)
    n == 0 && return -1
    n == 1 && return 0

    differences = Float64.(points[2:end, :]) .- Float64.(points[1:1, :])
    return rank(differences; atol=atol, rtol=0.0)
end

function validate_deterministic_vertices(vertices)
    @assert size(vertices, 2) == 64

    for vertex in eachrow(vertices)
        @assert all(value -> value == 0 || value == 1, vertex)

        for context in 0:7
            block = (8*context + 1):(8*context + 8)
            @assert sum(vertex[block]) == 1
        end
    end

    return true
end

vertices_clockwise = generate_orientation(:clockwise)
vertices_counterclockwise = generate_orientation(:counterclockwise)

# `unique(...; dims=1)` removes the 64 local deterministic behaviors that occur
# in both orientations. Keeping duplicates would not change the LP, but it would
# obscure the actual vertex count and the geometry.
vertices_cycle = unique(
    vcat(vertices_clockwise, vertices_counterclockwise);
    dims=1,
)

clockwise_set = row_set(vertices_clockwise)
counterclockwise_set = row_set(vertices_counterclockwise)

validate_deterministic_vertices(vertices_clockwise)
validate_deterministic_vertices(vertices_counterclockwise)
validate_deterministic_vertices(vertices_cycle)

@assert size(vertices_clockwise, 1) == 4096
@assert size(vertices_counterclockwise, 1) == 4096
@assert length(intersect(clockwise_set, counterclockwise_set)) == 64
@assert size(vertices_cycle, 1) == 8128

@assert affine_dimension(vertices_clockwise) == 44
@assert affine_dimension(vertices_counterclockwise) == 44
@assert affine_dimension(vertices_cycle) == 50

println("Corrected cycle polytope")
println("  clockwise vertices:        ", size(vertices_clockwise, 1))
println("  counterclockwise vertices: ", size(vertices_counterclockwise, 1))
println("  intersection:              ", length(intersect(clockwise_set, counterclockwise_set)))
println("  distinct union:            ", size(vertices_cycle, 1))
println("  affine dimension:          ", affine_dimension(vertices_cycle))




Corrected cycle polytope
  clockwise vertices:        4096
  counterclockwise vertices: 4096
  intersection:              64
  distinct union:            8128
  affine dimension:          50


## 3. Published representatives of NS classes 27, 29, 43, and 44

The four targets are entered through the 26 correlators of a binary tripartite nonsignalling behavior. Their stored order is

$$
\begin{aligned}
&(A_0,A_1;\ B_0,B_1;\ C_0,C_1;\\
&AB_{00},AB_{01},AB_{10},AB_{11};\\
&AC_{00},AC_{01},AC_{10},AC_{11};\\
&BC_{00},BC_{01},BC_{10},BC_{11};\\
&ABC_{000},ABC_{001},ABC_{010},ABC_{011},
ABC_{100},ABC_{101},ABC_{110},ABC_{111}).
\end{aligned}
$$

These representatives are those reported in Table 1 of S. Pironio, J.-D. Bancal, and V. Scarani, *Extremal correlations of the tripartite no-signaling polytope*, J. Phys. A **44**, 065303 (2011), arXiv:1101.2477.

To convert correlators into probabilities, define $s_a=(-1)^a$, and analogously for $b$ and $c$. Then

$$
\begin{aligned}
p(abc\mid xyz)=\frac18[&1+s_aA_x+s_bB_y+s_cC_z
+s_as_bAB_{xy}+s_as_cAC_{xz}\\
&+s_bs_cBC_{yz}+s_as_bs_cABC_{xyz}].
\end{aligned}
$$

The implementation uses Julia rational numbers such as `1//3` throughout this conversion. Consequently, positivity, normalization, nonsignalling, and the later evaluations of rationalized inequalities can be checked exactly rather than up to floating-point tolerance.


In [7]:
#
# Correlator order:
#
#   A0,A1, B0,B1, C0,C1,
#   AB00,AB01,AB10,AB11,
#   AC00,AC01,AC10,AC11,
#   BC00,BC01,BC10,BC11,
#   ABC000,ABC001,ABC010,ABC011,ABC100,ABC101,ABC110,ABC111.
#
# These representatives agree with Table 1 of:
# S. Pironio, J.-D. Bancal, and V. Scarani,
# "Extremal correlations of the tripartite no-signaling polytope",
# J. Phys. A 44, 065303 (2011), arXiv:1101.2477.

const NS_CORRELATORS = Dict(
    27 => Rational{Int}[
        1//2, 0, 0, 0, 0, 0,
        1//2, 1//2, 0, 0,
        1//2, 1//2, 0, 0,
        0, 0, 0, 0,
        -1//2, -1//2, -1//2, -1//2, 1, 1, 1, -1,
    ],
    29 => Rational{Int}[
        1//3, 1//3, 1//3, 1//3, 1//3, 1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1, -1//3, -1//3, -1//3,
        1//3, 1//3, 1//3, -1, 1//3, -1, -1, -1,
    ],
    43 => Rational{Int}[
        1//7, 1//7, 1//7, 1//7, 1//7, 1//7,
        3//7, -1//7, -1//7, -1//7,
        3//7, -1//7, -1//7, -1//7,
        -1//7, -1//7, -1//7, -1//7,
        1//7, 5//7, 5//7, -1, -1, -1, -1, -1,
    ],
    44 => Rational{Int}[
        0, 0, 0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        0, 0, 0, 0,
        1, 1, 1, 1, 1, 1, 1, -1,
    ],
)

outcome_sign(bit::Int) = bit == 0 ? 1 : -1

"""Convert the 26 NS correlators to the canonical 64 probabilities exactly."""
function correlators_to_probabilities(correlators)
    @assert length(correlators) == 26
    q = Rational{Int}.(correlators)
    probabilities = fill(0//1, 64)

    A = q[1:2]
    B = q[3:4]
    C = q[5:6]
    AB = q[7:10]
    AC = q[11:14]
    BC = q[15:18]
    ABC = q[19:26]

    for (x, y, z) in TRIPLES, (a, b, c) in TRIPLES
        sa = outcome_sign(a)
        sb = outcome_sign(b)
        sc = outcome_sign(c)

        probabilities[prob_index(a, b, c, x, y, z)] = (
            1 +
            sa*A[x + 1] +
            sb*B[y + 1] +
            sc*C[z + 1] +
            sa*sb*AB[2*x + y + 1] +
            sa*sc*AC[2*x + z + 1] +
            sb*sc*BC[2*y + z + 1] +
            sa*sb*sc*ABC[4*x + 2*y + z + 1]
        ) / 8
    end

    return probabilities
end

const NS_TARGETS = Dict(
    class => correlators_to_probabilities(NS_CORRELATORS[class])
    for class in CLASSES
)




Dict{Int64, Vector{Rational{Int64}}} with 4 entries:
  29 => [2//3, 0, 0, 0, 0, 0, 0, 1//3, 1//3, 1//3  …  0, 0, 0, 1//3, 1//3, 0, 1…
  43 => [2//7, 1//7, 1//7, 0, 0, 1//7, 1//7, 1//7, 2//7, 1//7  …  0, 1//7, 0, 2…
  27 => [1//4, 1//4, 1//4, 0, 0, 0, 0, 1//4, 1//4, 1//4  …  1//4, 0, 0, 1//4, 1…
  44 => [1//4, 0, 0, 1//4, 0, 1//4, 1//4, 0, 1//4, 0  …  1//4, 0, 0, 1//4, 1//4…

## 4. Exact target checks: positivity, normalization, nonsignalling, and extremality

Before using a target in an LP, this cell verifies the defining probabilistic and nonsignalling conditions directly in the 64-coordinate representation:

- every probability is nonnegative;
- the eight probabilities for each fixed input triple $(x,y,z)$ sum to 1;
- the $AB$ marginal is independent of Charlie's input $z$;
- the $AC$ marginal is independent of Bob's input $y$;
- the $BC$ marginal is independent of Alice's input $x$.

It then independently checks that each representative is a **vertex** of the nonsignalling polytope. The normalization and nonsignalling equalities are assembled as a (redundant) $56\times64$ matrix. Its rank is 38, hence the nonsignalling affine space has dimension

$$
64-38=26.
$$

At a given target, every zero probability saturates one positivity inequality. The target is extremal when the equality rows together with these active positivity rows have rank 64: there is then no nonzero feasible direction that keeps all active constraints saturated. The printed `active rank = 64 (NS vertex)` is therefore a geometric check, not merely a lookup of the class label.

All these tests use exact target probabilities. A failed assertion indicates an inconsistent representative, correlator order, probability order, or nonsignalling implementation and should be investigated before proceeding.


In [9]:

function validate_nonsignalling_behavior(probabilities)
    @assert length(probabilities) == 64
    @assert all(probability -> probability >= 0, probabilities)

    # Normalization for every input triple.
    for (x, y, z) in TRIPLES
        @assert sum(
            probabilities[prob_index(a, b, c, x, y, z)]
            for (a, b, c) in TRIPLES
        ) == 1
    end

    # AB marginal independent of z.
    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, y, 0)] for c in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, y, 1)] for c in 0:1)
        @assert lhs == rhs
    end

    # AC marginal independent of y.
    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, x, 0, z)] for b in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, x, 1, z)] for b in 0:1)
        @assert lhs == rhs
    end

    # BC marginal independent of x.
    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        lhs = sum(probabilities[prob_index(a, b, c, 0, y, z)] for a in 0:1)
        rhs = sum(probabilities[prob_index(a, b, c, 1, y, z)] for a in 0:1)
        @assert lhs == rhs
    end

    return true
end

"""
Return a (redundant) matrix for normalization and tripartite NS equalities.
Its rank must be 38, leaving the familiar 26-dimensional NS affine space.
"""
function nonsignalling_equality_matrix()
    equalities = zeros(Float64, 56, 64)
    row = 0

    for (x, y, z) in TRIPLES
        row += 1
        for (a, b, c) in TRIPLES
            equalities[row, prob_index(a, b, c, x, y, z)] = 1
        end
    end

    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        row += 1
        for c in 0:1
            equalities[row, prob_index(a, b, c, x, y, 0)] += 1
            equalities[row, prob_index(a, b, c, x, y, 1)] -= 1
        end
    end

    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        row += 1
        for b in 0:1
            equalities[row, prob_index(a, b, c, x, 0, z)] += 1
            equalities[row, prob_index(a, b, c, x, 1, z)] -= 1
        end
    end

    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        row += 1
        for a in 0:1
            equalities[row, prob_index(a, b, c, 0, y, z)] += 1
            equalities[row, prob_index(a, b, c, 1, y, z)] -= 1
        end
    end

    @assert row == 56
    return equalities
end

const NS_EQUALITIES = nonsignalling_equality_matrix()
@assert rank(NS_EQUALITIES; atol=1e-9, rtol=0.0) == 38

"""
Check extremality in the NS polytope using the rank of the NS equalities plus
the positivity inequalities saturated by the target.
"""
function ns_active_rank(probabilities)
    zero_coordinates = findall(iszero, probabilities)
    coordinate_rows = Matrix{Float64}(I, 64, 64)[zero_coordinates, :]
    active_matrix = vcat(NS_EQUALITIES, coordinate_rows)
    return rank(active_matrix; atol=1e-9, rtol=0.0)
end

println("\nTarget checks")
for class in CLASSES
    target = NS_TARGETS[class]
    validate_nonsignalling_behavior(target)
    active_rank = ns_active_rank(target)
    @assert active_rank == 64

    @printf(
        "  class %2d: support = %2d, active rank = %2d (NS vertex)\n",
        class,
        count(value -> !iszero(value), target),
        active_rank,
    )
end





Target checks
  class 27: support = 32, active rank = 64 (NS vertex)
  class 29: support = 23, active rank = 64 (NS vertex)
  class 43: support = 36, active rank = 64 (NS vertex)
  class 44: support = 32, active rank = 64 (NS vertex)


## 5. Separation LP for the corrected full cycle polytope

For each nonsignalling target $q$, the code searches for a nonnegative coefficient vector $c\in\mathbb{R}^{64}$ by solving

$$
\begin{aligned}
\text{minimize}\quad & c\cdot q\\
\text{subject to}\quad & c\cdot v\geq 1
&&\text{for all }v\in\mathcal{V}_{\mathrm{cycle}},\\
& c_j\geq0
&&\text{for }j=1,\ldots,64.
\end{aligned}
$$

The bound is normalized to 1. Since a linear functional attains its minimum over a polytope at a vertex, imposing the first constraint on all 8128 deterministic vertices proves

$$
c\cdot p\geq1
\qquad\forall p\in\mathcal{C}_{\mathrm{cycle}}.
$$

Therefore an optimal target value below 1 is a certificate that $q$ lies outside the cycle polytope. The LP is constrained by the corrected **full union**, not by one orientation alone. After solving, the code also reports the minima on the clockwise and counterclockwise subsets separately; both must respect the bound.

GLPK returns floating-point coefficients. The code reconstructs simple rational coefficients, reevaluates every vertex exactly, and requires the exact classical minimum to be 1. It then studies the vertices saturating $c\cdot v=1$. Because the full cycle polytope has affine dimension 50, a supporting inequality is facet-defining precisely when the saturated face has affine dimension 49.

Finally, the exact target values

| class | $c\cdot q$ |
|---:|---:|
| 27 | $3/4$ |
| 29 | $1/2$ |
| 43 | $5/7$ |
| 44 | $0$ |

are asserted as independent regression values. They do **not** replace the validity checks on both orientations: a scalar optimum can remain unchanged even when an LP was built from the wrong vertex set, whereas the associated optimizing coefficient vector need not remain valid.


In [11]:
#
# For each target q, solve
#
#   minimize    c . q
#   subject to  c . v >= 1    for every one of the 8128 cycle vertices,
#               c >= 0.
#
# A value below 1 gives the valid separating inequality c . p >= 1.

function rationalize_coefficients(coefficients; tol=1e-8)
    return [
        abs(value) <= tol ? 0//1 : rationalize(Int, value; tol=tol)
        for value in coefficients
    ]
end

function solve_separator(
    class::Int,
    target,
    vertices_cycle,
    vertices_clockwise,
    vertices_counterclockwise;
    tol=1e-8,
    verbose=false,
)
    n_vertices, dimension = size(vertices_cycle)
    @assert dimension == length(target) == 64

    model = Model(GLPK.Optimizer)
    verbose || set_silent(model)

    @variable(model, coefficients[1:dimension] >= 0)
    @objective(
        model,
        Min,
        sum(Float64(target[j]) * coefficients[j] for j in 1:dimension),
    )
    @constraint(
        model,
        classical[i=1:n_vertices],
        sum(vertices_cycle[i, j] * coefficients[j] for j in 1:dimension) >= 1,
    )

    optimize!(model)
    status = termination_status(model)
    status == MOI.OPTIMAL || error("LP for class $class ended with status $status")

    coefficients_float = value.(coefficients)
    coefficients_exact = rationalize_coefficients(coefficients_float; tol=tol)

    values_cycle = Float64.(vertices_cycle) * coefficients_float
    values_clockwise = Float64.(vertices_clockwise) * coefficients_float
    values_counterclockwise = Float64.(vertices_counterclockwise) * coefficients_float

    exact_cycle_values = [
        sum(
            coefficients_exact[j] * Int(vertices_cycle[i, j])
            for j in 1:dimension
        )
        for i in 1:n_vertices
    ]
    exact_target_value = sum(
        coefficients_exact[j] * target[j]
        for j in 1:dimension
    )

    exact_classical_minimum = minimum(exact_cycle_values)
    exact_classical_minimum == 1 || error(
        "Rational reconstruction failed for class $class: exact minimum is " *
        "$exact_classical_minimum instead of 1.",
    )

    saturating_indices = findall(==(1//1), exact_cycle_values)
    face_dimension = affine_dimension(vertices_cycle[saturating_indices, :])

    return (
        class=class,
        status=status,
        coefficients=coefficients_float,
        exact_coefficients=coefficients_exact,
        solver_target_value=objective_value(model),
        exact_target_value=exact_target_value,
        classical_minimum=minimum(values_cycle),
        clockwise_minimum=minimum(values_clockwise),
        counterclockwise_minimum=minimum(values_counterclockwise),
        saturation_count=length(saturating_indices),
        face_dimension=face_dimension,
        is_facet=(face_dimension == affine_dimension(vertices_cycle) - 1),
    )
end

separator_results = Dict{Int, Any}()

for class in CLASSES
    separator_results[class] = solve_separator(
        class,
        NS_TARGETS[class],
        vertices_cycle,
        vertices_clockwise,
        vertices_counterclockwise,
    )
end

# Independent expected values. These assertions guard against future changes in
# coordinate order, target conversion, or vertex generation.
const EXPECTED_TARGET_VALUES = Dict(
    27 => 3//4,
    29 => 1//2,
    43 => 5//7,
    44 => 0//1,
)

for class in CLASSES
    @assert separator_results[class].exact_target_value == EXPECTED_TARGET_VALUES[class]
end

println("\nCorrected separation results")
println("class  target   min(CW)   min(CCW)  saturating  face-dim  facet")
for class in CLASSES
    result = separator_results[class]
    @printf(
        "%5d  %7s   %7.4f   %8.4f  %10d  %8d  %s\n",
        class,
        string(result.exact_target_value),
        result.clockwise_minimum,
        result.counterclockwise_minimum,
        result.saturation_count,
        result.face_dimension,
        string(result.is_facet),
    )
end





Corrected separation results
class  target   min(CW)   min(CCW)  saturating  face-dim  facet
   27     3//4    1.0000     1.0000         119        49  true
   29     1//2    1.0000     1.0000          90        44  false
   43     5//7    1.0000     1.0000         508        49  true
   44     0//1    1.0000     1.0000         508        49  true


## 6. Inequality coefficient matrices

The final cell reshapes each exact 64-component coefficient vector into an $8\times8$ matrix for reading and reuse. Both axes follow the binary lexicographic order

$$
000,001,010,011,100,101,110,111.
$$

- Row `xyz` fixes the input triple.
- Column `abc` fixes the output triple.
- The entry in that row and column is the coefficient multiplying $p(abc\mid xyz)$.

Thus, if the displayed matrix for a class is denoted by $C$, its inequality is

$$
\sum_{x,y,z,a,b,c}
C_{xyz,abc}\,p(abc\mid xyz)\geq1.
$$

The matrices contain exact rational numbers and are generated from the coefficients that have already passed the full-vertex validity check. They are the appropriate first-stage outputs to carry into later quantum-bound and see-saw analyses.


In [13]:
# Rows and columns both follow 000,001,010,011,100,101,110,111.
# Rows label inputs (x,y,z), columns label outputs (a,b,c).

coefficient_matrix(coefficients) =
    Matrix(permutedims(reshape(coefficients, 8, 8)))

inequality_matrices = Dict(
    class => coefficient_matrix(separator_results[class].exact_coefficients)
    for class in CLASSES
)

println("\nExact coefficient matrices")
for class in CLASSES
    println("\nClass $class:  c . p >= 1")
    display(inequality_matrices[class])
end



Exact coefficient matrices

Class 27:  c . p >= 1


8×8 Matrix{Rational{Int64}}:
 1//2  1//2  1//2   1    1//2  1  1  0
  0     0     0    1//2  1//2  1  1  0
  0     0     0    1//2  1//2  1  1  0
 1//2  1//2  1//2   1    1//2  1  1  0
  0     1     1     0    1//2  0  0  1
  0     1     1     0    1//2  0  0  1
  0     1     1     0    1//2  0  0  1
 1//2   0     0     1     0    1  1  0


Class 29:  c . p >= 1


8×8 Matrix{Rational{Int64}}:
  0    1//2  1//2   1    1//2   1     1    0
  0    1//2   1     1     1     1     0    1
  0     1    1//2   1     1     0     1    1
 1//2   0     0     0     0     1     1    0
  0     1     1     0    1//2   1     1    1
 1//2   0     0     1     0     0     1    0
 1//2   0     0     1     0     1     0    0
  1     0     0    1//2   0    1//2  1//2  1


Class 43:  c . p >= 1


8×8 Matrix{Rational{Int64}}:
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 1  0  0  1  0  1  1  0
 1  0  0  1  0  1  1  0
 1  0  0  1  0  1  1  0
 1  0  0  1  0  1  1  0
 1  0  0  1  0  1  1  0


Class 44:  c . p >= 1


8×8 Matrix{Rational{Int64}}:
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 0  1  1  0  1  0  0  1
 1  0  0  1  0  1  1  0